In [1]:
!pip install jiwer torchtext sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.0 MB/s eta 0:00:00


In [2]:
# !pip install googletrans==4.0.0-rc1
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00


In [3]:
import os
import torch
import re
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from google.colab import drive

In [4]:
# =========================
# DATA
# =========================

# Mount Drive
drive.mount('/content/drive')

# Paths Configuration
BASE_PATH = "/content/drive/MyDrive"
ANNOTATIONS_CSV = f"{BASE_PATH}/annotations/manual/PHOENIX-2014-T.train.corpus.csv"
# KEYPOINTS_DIR = f"{BASE_PATH}/holistic/train/"
TRAIN_DATA_PICKLE = f"{BASE_PATH}/Phoenix-2014.train"
TEST_DATA_PICKLE = f"{BASE_PATH}/Phoenix-2014.test"
DEV_DATA_PICKLE = f"{BASE_PATH}/Phoenix-2014.dev"

# Model Path
MODEL_PATH = "/content/drive/MyDrive/MSKA-SLR/best.pth"


Mounted at /content/drive


In [5]:
# =========================
# TRAINING SETTINGS
# =========================

MAX_FRAMES = 150
MAX_TEXT_LEN = 50

BATCH_SIZE = 4

# =========================
# MODEL (SIMPLIFIED)
# =========================

INPUT_DIM = 399
MODEL_DIM = 256
NUM_HEADS = 4
NUM_LAYERS = 3
DROPOUT = 0.3

# =========================
# OPTIMIZATION
# =========================

LEARNING_RATE = 2e-4
NUM_EPOCHS = 70

# =========================
# DECODING
# =========================

BEAM_SIZE = 5
LENGTH_PENALTY = 1.0

# =========================
# TOKENS
# =========================

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

# =========================
# CHECKPOINTS
# =========================

# CHECKPOINT_DIR = "/content/checkpoints"

In [6]:
class CleanSLTTokenizer:
    """
    Hybrid tokenizer for Pose-to-Text SLT (PHOENIX-style)
    - cleaner than naive split
    - not dependent on MBART
    - upgradeable later to BPE/SentencePiece
    """
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.pad, self.sos, self.eos, self.unk = "<pad>", "<sos>", "<eos>", "<unk>"
        self.vocab = {self.pad: 0, self.sos: 1, self.eos: 2, self.unk: 3}
        self.inv_vocab = {}

    def normalize(self, text):
        text = text.lower()
        text = re.sub(r"[^a-zäöüß0-9\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def build_vocab(self, sentences):
        counter = Counter()
        for s in sentences:
            s = self.normalize(s)
            counter.update(s.split())
        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.vocab:
                self.vocab[word] = len(self.vocab)
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        text = self.normalize(text)
        tokens = [self.vocab[self.sos]]
        for w in text.split():
            tokens.append(self.vocab.get(w, self.vocab[self.unk]))
        tokens.append(self.vocab[self.eos])
        return tokens

    def decode(self, tokens):
        words = [self.inv_vocab.get(t, self.unk) for t in tokens if t > 3 and t != self.vocab[self.eos]]
        return " ".join(words)

In [7]:
import torch
import pickle
import numpy as np
from torch.utils.data import Dataset

class SLTPoseDataset(Dataset):
    """
    Final Clean Version: No external directories needed.
    Everything is loaded from the Pickle file.
    """
    def __init__(self, file_path, tokenizer, max_frames=150, max_text_len=50, input_dim=399):
        self.tokenizer = tokenizer
        self.max_frames = max_frames
        self.max_text_len = max_text_len
        self.input_dim = input_dim

        # Load the centralized dictionary
        with open(file_path, "rb") as f:
            full_data = pickle.load(f)

        # Convert dictionary to indexable list
        raw_samples = list(full_data.values()) if isinstance(full_data, dict) else full_data

        self.samples = []
        print(f"🔍 Processing {len(raw_samples)} samples from pickle...")

        for s in raw_samples:
            # Using the key we found in the diagnostic: 'keypoint'
            if s.get('keypoint') is not None:
                self.samples.append(s)

        print(f"✅ Ready! Loaded {len(self.samples)} valid samples.")

    def normalize_pose(self, pose):
        mean = torch.mean(pose, dim=0)
        std = torch.std(pose, dim=0) + 1e-6
        return (pose - mean) / std

    def pad_pose(self, pose):
        t, f = pose.shape
        if t < self.max_frames:
            padding = torch.zeros((self.max_frames - t, f))
            pose = torch.cat([pose, padding], dim=0)
        else:
            pose = pose[:self.max_frames, :]
        return pose

    def pad_text(self, tokens):
        if len(tokens) < self.max_text_len:
            tokens += [self.tokenizer.vocab["<pad>"]] * (self.max_text_len - len(tokens))
        return tokens[:self.max_text_len]

    def __len__(self):
        return len(self.samples)


    def temporal_resampling(self, pose, scale_range=(0.8, 1.2)):
      """
      Randomly scales the time dimension to simulate different signing speeds.
      """
      T, F = pose.shape
      # Randomly pick a new length within the scale range
      scale = np.random.uniform(scale_range[0], scale_range[1])
      new_len = int(T * scale)

      # Reshape for interpolation: (Batch, Channels, Time)
      pose = pose.unsqueeze(0).transpose(1, 2)
      # Linear interpolation to change speed
      pose = torch.nn.functional.interpolate(pose, size=new_len, mode='linear', align_corners=False)

      return pose.transpose(1, 2).squeeze(0)

    def random_rotation(self, pose, degree_range=(-10, 10)):
      """
      Applies a random rotation to the 3D keypoints.
      """
      # 1. Reshape from (T, 399) to (T, 133, 3)
      T, total_f = pose.shape
      pose = pose.view(T, -1, 3)

      # 2. Generate random angle
      angle = np.radians(np.random.uniform(degree_range[0], degree_range[1]))
      cos_a, sin_a = np.cos(angle), np.sin(angle)

      # 3. Z-axis rotation matrix
      rotation_matrix = torch.tensor([
          [cos_a, -sin_a, 0],
          [sin_a,  cos_a, 0],
          [0,      0,     1]
      ], dtype=torch.float32)

      # 4. Apply rotation and flatten back to (T, 399)
      pose = torch.matmul(pose, rotation_matrix)
      return pose.view(T, -1)

    # def __getitem__(self, idx):
    #     sample = self.samples[idx]

    #     # 1. Get Pose Data from 'keypoint'
    #     pose_raw = sample.get('keypoint')
    #     # 2. Get Text from 'gloss'
    #     sentence = sample.get('gloss', "")

    #     # Convert to Tensor safely
    #     pose = torch.as_tensor(pose_raw, dtype=torch.float32)

    #     # Flatten if data is [Frames, Joints, Channels]
    #     if pose.dim() > 2:
    #         pose = pose.flatten(1)

    #     # Standardize and Pad
    #     pose = self.pad_pose(self.normalize_pose(pose))

    #     # --- NEW: DATA AUGMENTATION (Add noise during training) ---
    #     if hasattr(self, 'training') and self.training:
    #         noise = torch.randn_like(pose) * 0.01 # 1% noise level
    #         pose = pose + noise

    #     # Tokenize
    #     tokens = self.pad_text(self.tokenizer.encode(sentence))

    #     return {
    #         "name": sample.get('name', str(idx)),
    #         "pose": pose,
    #         "text": torch.tensor(tokens, dtype=torch.long)
    #     }


    def __getitem__(self, idx):
      sample = self.samples[idx]
      pose_raw = sample.get('keypoint')
      sentence = sample.get('gloss', "")

      pose = torch.as_tensor(pose_raw, dtype=torch.float32)
      if pose.dim() > 2:
          pose = pose.flatten(1)

      # --- ADVANCED DATA AUGMENTATION ---
      if hasattr(self, 'training') and self.training:
          # 1. Temporal Resampling (Speed variation)
          pose = self.temporal_resampling(pose)

          # 2. Random Rotation (Viewpoint variation)
          pose = self.random_rotation(pose)

          # 3. Subtle Gaussian Noise (Existing)
          noise = torch.randn_like(pose) * 0.005
          pose = pose + noise

      # Standardize and Pad (Keep these at the end of augmentation)
      pose = self.pad_pose(self.normalize_pose(pose))
      tokens = self.pad_text(self.tokenizer.encode(sentence))

      return {
          "name": sample.get('name', str(idx)),
          "pose": pose,
          "text": torch.tensor(tokens, dtype=torch.long)
      }

In [8]:
def create_dataloader(dataset, batch_size=4, shuffle=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,      # Re-enable for speed since data is now in memory
        pin_memory=True,    # Speeds up data transfer to GPU
        drop_last=True
    )

# **CLEAN TEST PIPELINE**

In [12]:
# 1. Re-init Tokenizer from 'gloss'
tokenizer = CleanSLTTokenizer()
with open(TRAIN_DATA_PICKLE, "rb") as f:
    temp_data = pickle.load(f)
    all_glosses = [s.get('gloss', "") for s in (temp_data.values() if isinstance(temp_data, dict) else temp_data)]
    tokenizer.build_vocab(all_glosses)

# 2. Create Dataset (Now correctly matching the __init__ signature)
train_dataset = SLTPoseDataset(
    file_path=TRAIN_DATA_PICKLE,
    tokenizer=tokenizer,
    max_frames=MAX_FRAMES,
    input_dim=INPUT_DIM
)

# 3. Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 4. Final Batch Check
try:
    batch = next(iter(train_loader))
    print("\n🔥 SUCCESS! DATA PIPELINE IS LIVE.")
    print(f"Batch Pose Shape: {batch['pose'].shape}")
    print(f"First Gloss in Batch: {tokenizer.decode(batch['text'][0].tolist())}")
except Exception as e:
    print(f"\n❌ Error during Batch extraction: {e}")

🔍 Processing 5672 samples from pickle...
✅ Ready! Loaded 5672 valid samples.

🔥 SUCCESS! DATA PIPELINE IS LIVE.
Batch Pose Shape: torch.Size([4, 150, 399])
First Gloss in Batch: sued mild zehn bis vierzehn grad nord enorm kalt sinken ix sonne dabei


---------------

# **UTILS CELL**

In [13]:
import torch
import torch.nn as nn
import numpy as np
import math
import torch.nn.functional as F

# =============================================================================
# 1. POSITIONAL ENCODING (Deep spatial-temporal awareness)
# =============================================================================
class PositionalEncoding(nn.Module):
    """
    Injects information about the relative or absolute position of the tokens
    in the sequence. Based on sine and cosine functions of different frequencies.
    """
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a long enough positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# =============================================================================
# 2. MASKING GENERATOR (Critical for Transformer attention)
# =============================================================================
class MaskGenerator:
    """
    Generates masks for padding and look-ahead (causal) attention.
    """
    @staticmethod
    def make_pad_mask(seq, pad_idx):
        # seq: (batch, seq_len)
        mask = (seq != pad_idx).unsqueeze(1).unsqueeze(2)
        return mask # (batch, 1, 1, seq_len)

    @staticmethod
    def make_subsequent_mask(size):
        """
        Creates a causal mask for decoder self-attention.
        Shape for PyTorch: (size, size)
        """
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        # PyTorch Transformer expects True for positions to mask (future tokens)
        return mask

# =============================================================================
# 3. LABEL SMOOTHING (Advanced Regularization)
# =============================================================================
class LabelSmoothingLoss(nn.Module):
    """
    Label smoothing loss to improve generalization and prevent over-fitting.
    """
    def __init__(self, size, padding_idx, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        return self.criterion(x, true_dist.detach())

# =============================================================================
# 4. BEAM SEARCH DECODER (Inference Logic)
# =============================================================================
def beam_search_decode(model, pose_data, tokenizer, beam_size=5, max_len=50):
    """
    Advanced Beam Search for high-quality translation.
    """
    model.eval()
    device = pose_data.device
    batch_size = pose_data.size(0)

    with torch.no_grad():
        # Encode visual features
        rec_out = model.recognition_network(pose_data)
        memory = model.vl_mapper(rec_out)

        # Initialize sequences (batch, beam, seq_len)
        # For simplicity, we implement a single batch beam search here
        start_symbol = tokenizer.vocab["<sos>"]
        end_symbol = tokenizer.vocab["<eos>"]

        # This is a complex logic; usually provided by a dedicated class
        # For now, Greedy is fine for testing, but for production,
        # we'll use this placeholder to call the model's generate method.
        return model.generate(pose_data, max_len=max_len)

# =============================================================================
# 5. CONVOLUTIONAL LAYER FOR POSE (Temporal Pooling)
# =============================================================================
class TemporalConv(nn.Module):
    """
    A 1D convolutional layer to reduce temporal resolution of pose sequences,
    mimicking the 'Recognition' behavior in MSKA.
    """
    def __init__(self, input_size, output_size, kernel_size=3, stride=1, padding=1):
        super(TemporalConv, self).__init__()
        self.conv = nn.Conv1d(input_size, output_size, kernel_size, stride, padding)
        self.bn = nn.BatchNorm1d(output_size)
        self.activation = nn.ReLU()

    def forward(self, x):
        # x: (batch, time, features) -> needs permute for Conv1d
        x = x.permute(0, 2, 1)
        x = self.activation(self.bn(self.conv(x)))
        return x.permute(0, 2, 1) # back to (batch, time, features)

In [14]:
import torch
import torch.nn as nn

class SpatialEmbedding(nn.Module):
    def __init__(self, hidden_size=512):
        super(SpatialEmbedding, self).__init__()

        # PHOENIX/MediaPipe 133 joints typically breakdown:
        # Face: 70 joints | Hands: 42 joints (21 each) | Body: 21 joints
        # (Check your specific extractor indices to match these)

        self.face_stream = nn.Sequential(
            nn.Linear(70 * 3, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_size // 4)
        )

        self.hand_stream = nn.Sequential(
            nn.Linear(42 * 3, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_size // 2)
        )

        self.body_stream = nn.Sequential(
            nn.Linear(21 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, hidden_size // 4)
        )

        # Final fusion to merge all streams back into the Transformer dimension
        self.fusion = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # x shape: (Batch, Time, 399)
        # Slicing the 399 features based on MediaPipe indices
        # Adjust indices [0:210], [210:336], [336:399] to match your data format
        face_feat = x[:, :, 0:210]
        hand_feat = x[:, :, 210:336]
        body_feat = x[:, :, 336:399]

        f_out = self.face_stream(face_feat)
        h_out = self.hand_stream(hand_feat)
        b_out = self.body_stream(body_feat)

        # Concatenate along the feature dimension
        combined = torch.cat([f_out, h_out, b_out], dim=-1)
        return self.dropout(self.fusion(combined))

In [15]:
import torch.nn.functional as F

class Recognition(nn.Module):
    def __init__(self, input_dim=399, hidden_size=512, vocab_size=1000):
        super(Recognition, self).__init__()

        # NEW: Spatial Embedding Stream
        self.spatial_embedding = SpatialEmbedding(hidden_size)

        self.conv1d = nn.Conv1d(input_dim, hidden_size, kernel_size=3, stride=1, padding=1)
        # self.bn = nn.BatchNorm1d(hidden_size)
        # self.relu = nn.ReLU()

        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=8, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # The output size must be vocab_size + 1 (for the CTC blank token)
        self.visual_head = nn.Linear(hidden_size, vocab_size + 1)

    def forward(self, x):
        # 1. Spatial Processing
        x = self.spatial_embedding(x) # (B, T, 512)

        # 2. Temporal Processing (CNN)
        x = x.transpose(1, 2)
        # x = self.relu(self.bn(self.conv1d(x)))
        x = x.transpose(1, 2)

        memory = self.encoder(x)

        logits = self.visual_head(memory) # Shape: (Batch, Time, Vocab+1)

        # CTC needs Log-Softmax
        log_probs = F.log_softmax(logits, dim=-1)

        return {"video_features": memory, "gloss_logits": log_probs}

In [16]:
class VLMapper(nn.Module):
    """
    The bridge between Visual Feature Space and Linguistic Latent Space.
    Ensures the Encoder output matches the Transformer Decoder input dimensions.
    """
    def __init__(self, in_features=512, out_features=512):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.LayerNorm(out_features),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, visual_outputs):
        # Extract features from recognition network outputs
        x = visual_outputs["video_features"]
        return self.projection(x)

In [17]:
class TranslationNetwork(nn.Module):
    """
    Transformer Decoder based Translation Network.
    Converts mapped visual features into natural language text tokens.
    """
    def __init__(self, input_dim=512, vocab_size=5000):
        super().__init__()
        self.input_dim = input_dim

        # Token Embedding and Positional Encoding (from Utils Cell)
        self.embedding = nn.Embedding(vocab_size, input_dim)
        self.pos_encoding = PositionalEncoding(input_dim)

        # Standard Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(d_model=input_dim, nhead=8, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=3)

        self.output_layer = nn.Linear(input_dim, vocab_size)

    def forward(self, input_feature, target_ids, tgt_mask=None, memory_mask=None):
        # Embed target tokens and add positional information
        tgt_emb = self.pos_encoding(self.embedding(target_ids))

        # Decoding process
        output = self.decoder(
            tgt=tgt_emb,
            memory=input_feature,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=memory_mask
        )

        logits = self.output_layer(output)
        return {"translation_logits": logits}

In [18]:
class SignLanguageModel(nn.Module):
    """
    Top-level Wrapper for Multi-Stream Keypoint Attention (MSKA) Tasks.
    Integrates Recognition, Mapping, and Translation modules.
    """
    def __init__(self, input_dim=399, model_dim=512, vocab_size=5000):
        super().__init__()

        # Initialize Sub-networks
        self.recognition_network = Recognition(input_dim=input_dim, hidden_size=model_dim)
        self.vl_mapper = VLMapper(in_features=model_dim, out_features=model_dim)
        self.translation_network = TranslationNetwork(input_dim=model_dim, vocab_size=vocab_size)

        self.task = 'S2T' # Sign-to-Text mode

      # Update the forward inside SignLanguageModel class:
    def forward(self, pose_data, target_ids=None, tgt_mask=None, **kwargs):

        """
        Forward pass for training.
        pose_data: (batch, time, 399)
        target_ids: (batch, seq_len) - Ground truth tokens for Teacher Forcing
        """
        recognition_outputs = self.recognition_network(pose_data)
        mapped_feature = self.vl_mapper(recognition_outputs)

        if target_ids is not None:
            # Pass the tgt_mask here!
            translation_outputs = self.translation_network(
                mapped_feature, target_ids, tgt_mask=tgt_mask
            )
            return {**recognition_outputs, **translation_outputs}

        return recognition_outputs

    def generate(self, pose_data, beam_size=5, max_len=50, start_idx=1, end_idx=2):
        """
        Final Professional Beam Search implementation to improve BLEU and variety.
        """
        self.eval()
        device = pose_data.device
        batch_size = pose_data.size(0)

        with torch.no_grad():
            rec_out = self.recognition_network(pose_data)
            memory = self.vl_mapper(rec_out) # (batch, time, dim)

            # We will perform beam search for each item in batch
            all_generated = []
            for b in range(batch_size):
                # Initial memory for this sample
                mem = memory[b:b+1]

                # Top sequences: (score, sequence)
                beams = [(0, torch.tensor([start_idx], device=device))]

                for _ in range(max_len):
                    new_beams = []
                    for score, seq in beams:
                        if seq[-1].item() == end_idx:
                            new_beams.append((score, seq))
                            continue

                        # Predict next token
                        out = self.translation_network(mem, seq.unsqueeze(0))
                        logits = out["translation_logits"][:, -1, :]
                        probs = torch.log_softmax(logits, dim=-1)

                        # Get top K candidates
                        top_probs, top_indices = probs.topk(beam_size)

                        for i in range(beam_size):
                            new_score = score + top_probs[0, i].item()
                            new_seq = torch.cat([seq, top_indices[0, i].unsqueeze(0)])
                            new_beams.append((new_score, new_seq))

                    # Keep top beam_size sequences
                    beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]

                    # Stop if best sequence is EOS
                    if beams[0][1][-1].item() == end_idx:
                        break

                all_generated.append(beams[0][1])

            # Pad and stack results to return a tensor
            max_gen_len = max(len(x) for x in all_generated)
            final_tensor = torch.zeros(batch_size, max_gen_len, device=device, dtype=torch.long)
            for i, gen in enumerate(all_generated):
                final_tensor[i, :len(gen)] = gen

            return final_tensor

## **EVALUATION UTILS CELL**

In [19]:
import torch
import numpy as np
from jiwer import wer
import sacrebleu

class SLTEvaluator:
    """
    Comprehensive Evaluation Suite for Sign Language Translation.
    Uses sacrebleu for BLEU and jiwer for WER.
    """
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def compute_bleu(self, hypotheses, references):
        """Calculates BLEU score using SacreBLEU."""
        if not hypotheses or not references:
            return 0.0
        # sacrebleu expects a list of hypotheses and a list of lists for references
        bleu = sacrebleu.corpus_bleu(hypotheses, [references])
        return bleu.score

    def compute_wer(self, hypotheses, references):
        """Calculates Word Error Rate (WER)."""
        if not hypotheses or not references:
            return 100.0
        # Ensure all items are strings
        hyp_strs = [str(h) for h in hypotheses]
        ref_strs = [str(r) for r in references]
        try:
            error_rate = wer(ref_strs, hyp_strs)
            return error_rate * 100
        except Exception as e:
            return 100.0

    def get_all_metrics(self, hyps, refs):
        """Returns a dictionary of all relevant SLT metrics."""
        return {
            "bleu": self.compute_bleu(hyps, refs),
            "wer": self.compute_wer(hyps, refs)
        }

In [20]:
def evaluate_model(model, data_loader, evaluator, device):
    model.eval()
    all_hypotheses = []
    all_references = []

    with torch.no_grad():
        for batch in data_loader:
            poses = batch["pose"].to(device)
            targets = batch["text"] # Ground truth indices

            # 1. Generate text using the greedy/beam search method
            # start_idx=1 (SOS), end_idx=2 (EOS)
            generated_indices = model.generate(poses, max_len=50, start_idx=1, end_idx=2)

            # 2. Decode indices back to strings
            for i in range(generated_indices.size(0)):
                hyp_str = evaluator.tokenizer.decode(generated_indices[i].tolist())
                ref_str = evaluator.tokenizer.decode(targets[i].tolist())

                all_hypotheses.append(hyp_str)
                all_references.append(ref_str)

    # 3. Calculate scores
    metrics = evaluator.get_all_metrics(all_hypotheses, all_references)
    return metrics, all_hypotheses, all_references

In [21]:
from deep_translator import GoogleTranslator

def evaluate_and_translate(model, data_loader, evaluator, device, target_lang='en'):
    """
    Evaluates the model and translates German Sign Language (GSL) to English.
    Using deep_translator for better stability.
    """
    model.eval()
    # Initialize the translator (Source: German, Target: English)
    translator = GoogleTranslator(source='de', target=target_lang)
    results = []

    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            if i >= 3: break

            poses = batch["pose"].to(device)
            targets = batch["text"]

            generated_indices = model.generate(poses, max_len=50)

            for j in range(generated_indices.size(0)):
                german_text = evaluator.tokenizer.decode(generated_indices[j].tolist())
                reference_text = evaluator.tokenizer.decode(targets[j].tolist())

                # Translation logic with deep_translator
                try:
                    if german_text.strip():
                        eng_translation = translator.translate(german_text)
                    else:
                        eng_translation = "[Empty Output]"

                    ref_translation = translator.translate(reference_text)
                except Exception as e:
                    eng_translation = f"[Translation Error]"
                    ref_translation = f"[Translation Error]"

                results.append({
                    "German (Model)": german_text,
                    "English (Model)": eng_translation,
                    "English (Ref)": ref_translation
                })

    return results

In [22]:
# 1. Update Settings based on our discovery
INPUT_DIM = 399 # We found this in your batch shape
VOCAB_SIZE = len(tokenizer.vocab)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Instantiate Model
model = SignLanguageModel(input_dim=INPUT_DIM, model_dim=512, vocab_size=VOCAB_SIZE).to(DEVICE)

# 3. Load Pre-trained Weights
def load_checkpoint(model,slt_path, device):
    print(f"📂 Loading weights from {slt_path}...")
    checkpoint = torch.load(slt_path, map_location=DEVICE)

    # MSKA weights often have a 'model' or 'state_dict' prefix
    state_dict = checkpoint.get('state_dict', checkpoint.get('model', checkpoint))

    model_dict = model.state_dict()

    # Load Decoder & Mapper (The Hard Match)
    if os.path.exists(slt_path):
        slt_state = torch.load(slt_path, map_location=device)
        slt_state = slt_state.get('model', slt_state.get('state_dict', slt_state))
        loaded_slt = 0

        print("🔍 Deep Scanning for Translation & Mapping Weights...")
        for k, v in slt_state.items():
            suffix = ".".join(k.split('.')[-2:])

            for m_k in model_dict.keys():
                if any(x in m_k.lower() for x in ["translation", "mapper"]) and \
                   v.size() == model_dict[m_k].size():

                    if k.split('.')[-1] == m_k.split('.')[-1]:
                        model_dict[m_k] = v
                        loaded_slt += 1
                        break

        print(f"✅ Decoder & Mapper loaded: {loaded_slt} layers.")

    model.load_state_dict(model_dict, strict=False)
    print("🚀 MASTER FUSION COMPLETE! Expert knowledge injected.")


# Re-run the loading
load_checkpoint(model, MODEL_PATH,DEVICE)
# Re-run the loading


📂 Loading weights from /content/drive/MyDrive/MSKA-SLR/best.pth...
🔍 Deep Scanning for Translation & Mapping Weights...
✅ Decoder & Mapper loaded: 42 layers.
🚀 MASTER FUSION COMPLETE! Expert knowledge injected.


Loss and Optimizer Setup

In [23]:
!pip install transformers -q
# --- OPTIMIZATION SETUP ---
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup

# Label Smoothing is preferred for NMT/SLT tasks
criterion = LabelSmoothingLoss(size=VOCAB_SIZE, padding_idx=PAD_IDX, smoothing=0.1).to(DEVICE)
ctc_criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True).to(DEVICE)

# AdamW is the standard for Transformers
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning rate scheduler (Warmup is critical for Transformers)
accumulation_steps = 4
total_steps = (len(train_loader) // accumulation_steps) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


The Professional Training Loop

In [24]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler, device, clip=1.0, lambda_ctc=0.2):
    model.train()
    epoch_loss = 0
    accumulation_steps = 4

    optimizer.zero_grad()

    for i, batch in enumerate(loader):
        poses = batch["pose"].to(device)
        targets = batch["text"].to(device)

        trg_input = targets[:, :-1]
        trg_y = targets[:, 1:]

        tgt_mask = MaskGenerator.make_subsequent_mask(trg_input.size(1)).to(device)

        batch_size = poses.size(0)
        input_lengths = torch.full(size=(batch_size,), fill_value=MAX_FRAMES, dtype=torch.long).to(device)
        target_lengths = (targets != PAD_IDX).sum(dim=1).to(device)

        output = model(poses, target_ids=trg_input, tgt_mask=tgt_mask)
        logits = output["translation_logits"]
        log_probs = F.log_softmax(logits, dim=-1)

        loss_trans = criterion(log_probs.contiguous().view(-1, log_probs.size(-1)),
                         trg_y.contiguous().view(-1))

        ctc_logits = output["gloss_logits"].transpose(0, 1)
        loss_ctc = ctc_criterion(ctc_logits, targets, input_lengths, target_lengths)

        total_loss = loss_trans + (lambda_ctc * loss_ctc)
        total_loss = total_loss / accumulation_steps

        total_loss.backward()

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += (total_loss.item() * accumulation_steps)

        if i % 200 == 0:
            print(f"  Batch {i}/{len(loader)} | Loss: {total_loss.item() * accumulation_steps:.4f}")

    return epoch_loss / len(loader)


In [25]:
def validate(model, loader, evaluator, device):
    """
    Runs evaluation using BLEU and WER metrics.
    """
    model.eval()
    metrics, hyps, refs = evaluate_model(model, loader, evaluator, device)
    return metrics

The Full Execution Engine

In [ ]:
# ==========================================
# FINAL TRAINING & AUTO-SAVE ENGINE
# ==========================================

import os

# 1. Configuration & Setup
best_bleu = -1.0  # We start with a very low value to capture the first improvement
evaluator = SLTEvaluator(tokenizer)

# Ensure the directory for saving weights exists
save_dir = os.path.join(BASE_PATH, "MSKA_Checkpoints")
os.makedirs(save_dir, exist_ok=True)

# 2. Create Validation Loader (Crucial for BLEU calculation)
print("📂 Preparing Validation Data...")
dev_dataset = SLTPoseDataset(DEV_DATA_PICKLE, tokenizer, max_frames=MAX_FRAMES, input_dim=INPUT_DIM)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"🚀 Starting Professional Training on {DEVICE}...")

# Lists for storing results
train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []
bleu_history = []

# 3. Main Loop
for epoch in range(NUM_EPOCHS):
    print(f"\n" + "="*30)
    print(f"      EPOCH {epoch+1} / {NUM_EPOCHS}")
    print("="*30)

    # --- PHASE 1: TRAINING ---
    # Update training mode for Augmentation
    train_dataset.training = True

    # train_one_epoch should now return both loss and accuracy
    # avg_train_loss, avg_train_acc = train_one_epoch(...)
    avg_train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, DEVICE)

    # We use a placeholder for train_acc if your function doesn't support it yet
    train_acc = (100 - avg_train_loss) # Placeholder logic, replace with real acc from train_one_epoch

    train_loss_history.append(avg_train_loss)
    train_acc_history.append(train_acc)

    # --- PHASE 2: VALIDATION ---
    train_dataset.training = False # Disable noise during validation

    # evaluate_model should return metrics, including val_loss and accuracy
    val_metrics, hyps, refs = evaluate_model(model, dev_loader, evaluator, DEVICE)

    current_bleu = val_metrics['bleu']
    current_wer = val_metrics['wer']
    val_loss = val_metrics.get('loss', 0.0) # Ensure your evaluate_model returns 'loss'
    val_acc = val_metrics.get('accuracy', 0.0) # Ensure your evaluate_model returns 'accuracy'

    bleu_history.append(current_bleu)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)

    print(f"✨ Validation Results:")
    print(f"   -> Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   -> Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"   -> BLEU Score: {current_bleu:.2f}")
    print(f"   -> WER Score:  {current_wer:.2f}")

    # Display a quick sample from the model output vs ground truth
    print(f"   -> Sample Output: {hyps[0]}")
    print(f"   -> Ground Truth:  {refs[0]}")

    # --- PHASE 3: SAVE BEST MODEL ---
    # We save based on BLEU improvement
    if current_bleu > best_bleu:
        best_bleu = current_bleu
        save_path = os.path.join(save_dir, "best_finetuned_model.pth")

        # Save a comprehensive checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'bleu': best_bleu,
            'wer': current_wer,
            'train_loss': avg_train_loss,
            'val_loss': val_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'vocab': tokenizer.vocab,
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
            'train_acc_history': train_acc_history,
            'val_acc_history': val_acc_history,
            'bleu_history': bleu_history
        }, save_path)

        print(f"💾 EXCELLENT! New Best Model Saved to: {save_path}")
    else:
        print(f"ℹ️ BLEU did not improve (Best remains: {best_bleu:.2f})")

print("\n" + "!"*30)
print(f"✅ TRAINING COMPLETE!")
print(f"🏆 Final Best BLEU: {best_bleu:.2f}")
print("!"*30)

📂 Preparing Validation Data...
🔍 Processing 540 samples from pickle...
✅ Ready! Loaded 540 valid samples.
🚀 Starting Professional Training on cpu...

      EPOCH 1 / 70
  Batch 0/1418 | Loss: 208.6110


In [ ]:
# # Test the translation
# print("🌍 Translating Model Results to English...")
# samples = evaluate_and_translate(model, dev_loader, evaluator, DEVICE)

# for idx, s in enumerate(samples):
#     print(f"\n--- Sample {idx+1} ---")
#     print(f"🇩🇪 German Output:  {s['German (Model)']}")
#     print(f"🇺🇸 English Output: {s['English (Model)']}")
#     print(f"✅ English Ref:    {s['English (Ref)']}")

In [ ]:
# import matplotlib.pyplot as plt

# def plot_training_results(train_losses, val_bleus):
#     """
#     Plot training loss and validation BLEU across all epochs.
#     """
#     epochs = range(1, len(train_losses) + 1)

#     plt.figure(figsize=(12, 5))

#     # Loss Plot
#     plt.subplot(1, 2, 1)
#     plt.plot(epochs, train_losses, 'r-o', label='Training Loss')
#     plt.title('Training Loss Progression')
#     plt.xlabel('Epochs')
#     plt.ylabel('Loss Value')
#     plt.grid(True)
#     plt.legend()

#     # BLEU Plot (Ensuring same dimension as epochs)
#     plt.subplot(1, 2, 2)
#     plt.plot(range(1, len(val_bleus) + 1), val_bleus, 'b-s', label='Validation BLEU')
#     plt.title('BLEU Score Progression')
#     plt.xlabel('Epochs')
#     plt.ylabel('BLEU Score')
#     plt.grid(True)
#     plt.legend()

#     plt.tight_layout()
#     plt.show()

# plot_training_results(train_loss_history, bleu_history)
